# 03 · Kvasir-SEG: EIoU vs AEIoU Thorough Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipun-taneja/amorphous-yolo/blob/main/notebooks/03_kvasir_eiou_vs_aeiou.ipynb)

## Abstract

This notebook is a focused companion to `02_full_loss_comparison.ipynb`.
Where notebook 02 benchmarks **all 7 IoU-based losses** on the DUO underwater dataset,
this notebook zooms in on a single research question:

> **Does AEIoU (Amorphous-EIoU) outperform EIoU on a second amorphous-object domain?**

We test this on **Kvasir-SEG** — 1 000 colonoscopy polyp images from the Simula
Research Lab. Polyps are biomedical objects with highly irregular, concave boundaries
and variable aspect ratios, making them an ideal second test case for the AEIoU
λ-rigidity hypothesis:

> *Amorphous labels are imprecise → down-weighting the shape penalty (λ < 1) should
> improve both accuracy and noise robustness.*

### What this notebook does
1. Downloads Kvasir-SEG and converts segmentation masks → YOLO bounding boxes
2. Builds three validation splits: **clean**, **low-noise** (σ=0.02), **high-noise** (σ=0.08)
3. Trains **33 models** (3 EIoU + 10 AEIoU λ values × 3 splits) using YOLO26n
4. Runs **7 quantitative analyses** and **3 validity checks**
5. Produces publication-ready figures saved to `experiments_kvasir/analysis/`

### Cross-dataset validation
If the optimal λ found here matches the optimal λ from notebook 02 (DUO), that is
strong evidence of a domain-agnostic setting for amorphous object detection.


## Section 1 · Environment Setup

**Requirements**
- Google Colab with T4 GPU (or better — A100 recommended for speed)
- ~2–3 hours total runtime for all 33 training runs on T4
- No API keys required — Kvasir-SEG downloads directly from Simula Research Lab

**Runtime estimate per run:** ~3–5 min on T4 (20 epochs, 640 px, nano model)


In [ ]:
# --- Install pinned dependencies
# ultralytics 8.4.9: confirmed working with yolo26n.pt and our monkey-patch
# wandb 0.24.1: optional experiment tracking (silently skipped if no API key)
!pip install --upgrade pip -q
!pip install -U "ultralytics==8.4.9" "wandb==0.24.1" -q
print("Dependencies installed.")


In [ ]:
# --- Idempotent git clone
# Safe to re-run: skips the clone if the repo is already present.
# The repo contains src/losses.py (EIoULoss, AEIoULoss) and the data/ yaml files.
import os, sys

REPO_PATH = "/content/amorphous-yolo"
if not os.path.exists(f"{REPO_PATH}/.git"):
    print("Cloning amorphous-yolo...")
    os.system(f"git clone https://github.com/nipun-taneja/amorphous-yolo.git {REPO_PATH}")
else:
    print("Repo already present — skipping clone.")

# Add project root to Python path so src.losses is importable
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# --- All experiment constants (single source of truth for this notebook)
import math, time
from pathlib import Path
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR  = Path("/content/amorphous-yolo")
# Kvasir-SEG dataset root; the zip extracts to kvasir-seg/images/ and kvasir-seg/masks/
DATASET_ROOT = PROJECT_DIR / "datasets" / "kvasir-seg"
# Separate experiments dir from DUO (notebook 02) to avoid run-name collisions
EXPERIMENTS  = PROJECT_DIR / "experiments_kvasir"
ANALYSIS_DIR = EXPERIMENTS / "analysis"
MANIFEST_PATH = EXPERIMENTS / "manifest.json"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# ── Google Drive persistence ───────────────────────────────────────────────────
DRIVE_ROOT        = Path("/content/drive/MyDrive/amorphous_yolo")
DRIVE_EXPERIMENTS = DRIVE_ROOT / "experiments_kvasir"
DRIVE_AVAILABLE   = False   # set to True by mount_drive() below

# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS   = 20     # Sufficient for convergence comparison at nano scale
IMGSZ    = 640    # Standard YOLO input resolution
DEVICE   = 0      # GPU 0; change to "cpu" for debugging
MODEL_PT = "yolo26n.pt"  # Nano model — fast iteration, still meaningful metrics

# ── Random seeds for statistical rigour ───────────────────────────────────────
# For quick development runs, keep SEEDS = [42].
# For publication, set SEEDS = [42, 123, 456] to get mean ± std across 3 seeds.
# Each seed produces a separate run: kvasir_yolo26n_{loss}_{split}_s{seed}_e{epochs}
SEEDS = [42]

# ── Standard baselines from src/losses.py ─────────────────────────────────────
# All 6 published IoU-family losses are compared against AEIoU.
# This ensures AEIoU is benchmarked against the FULL field, not just EIoU.
BASELINE_LOSS_NAMES = ["iou", "giou", "diou", "ciou", "eiou", "eciou", "siou", "wiou"]

# ── AEIoU rigidity grid ───────────────────────────────────────────────────────
# λ=0.1 -> nearly pure center-alignment loss (shape penalty down-weighted 90%)
# λ=0.5 -> moderate trust in polyp extent labels
# λ=1.0 -> full size penalty active (normalisation still differs from EIoU — target
#           dims vs enclosing dims — so AEIoU(λ=1) != EIoU; used as cross-check)
AEIOU_RIGIDITIES = [round(x * 0.1, 1) for x in range(1, 11)]

def _fmt_r(r):
    # Format rigidity float for use in run names: 0.3 -> '0p3'
    return str(r).replace(".", "p")

# Master list of all loss keys (baselines + AEIoU grid)
ALL_LOSS_KEYS = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]

# ── Dataset split configs ─────────────────────────────────────────────────────
SPLIT_CONFIGS = {
    "clean": PROJECT_DIR / "data" / "kvasir_seg.yaml",
    "low":   PROJECT_DIR / "data" / "kvasir_seg_low.yaml",
    "high":  PROJECT_DIR / "data" / "kvasir_seg_high.yaml",
}

# ── Visualisation palette ─────────────────────────────────────────────────────
PALETTE = {
    # Baselines — warm/neutral tones
    "iou":         "#888888",  # Grey — vanilla IoU
    "giou":        "#BC6C25",  # Brown — GIoU
    "diou":        "#606C38",  # Olive — DIoU
    "ciou":        "#DDA15E",  # Tan — CIoU (Ultralytics default)
    "eiou":        "#E63946",  # Red — EIoU
    "eciou":       "#9B2226",  # Dark red — ECIoU
    "siou":        "#6A0572",  # Violet — SIoU (Gevorgyan 2022)
    "wiou":        "#FF6B6B",  # Coral — WIoU (Tong 2023)
    # AEIoU — cool tones (blue-green gradient by λ)
    "aeiou_r0p1":  "#023E8A",  # Navy — λ=0.1
    "aeiou_r0p2":  "#0077B6",  # Blue — λ=0.2
    "aeiou_r0p3":  "#00B4D8",  # Cyan — λ=0.3
    "aeiou_r0p4":  "#48CAE4",  # Light cyan — λ=0.4
    "aeiou_r0p5":  "#90E0EF",  # Pale blue — λ=0.5
    "aeiou_r0p6":  "#2A9D8F",  # Teal — λ=0.6
    "aeiou_r0p7":  "#52B788",  # Green — λ=0.7
    "aeiou_r0p8":  "#74C69D",  # Light green — λ=0.8
    "aeiou_r0p9":  "#95D5B2",  # Pale green — λ=0.9
    "aeiou_r1p0":  "#6A4C93",  # Purple — λ=1.0
}

# ── Human-readable labels for plots ───────────────────────────────────────────
LOSS_LABELS = {
    "iou": "IoU", "giou": "GIoU", "diou": "DIoU", "ciou": "CIoU",
    "eiou": "EIoU", "eciou": "ECIoU", "siou": "SIoU", "wiou": "WIoU",
}
for r in AEIOU_RIGIDITIES:
    LOSS_LABELS[f"aeiou_r{_fmt_r(r)}"] = f"AEIoU {chr(955)}={r}"

n_baselines = len(BASELINE_LOSS_NAMES)
n_aeiou     = len(AEIOU_RIGIDITIES)
n_splits    = len(SPLIT_CONFIGS)
n_seeds     = len(SEEDS)
n_total     = (n_baselines + n_aeiou) * n_splits * n_seeds

print("Constants loaded.")
print(f"  Baselines:  {BASELINE_LOSS_NAMES}")
print(f"  AEIoU grid: {n_aeiou} values ({AEIOU_RIGIDITIES})")
print(f"  Splits:     {list(SPLIT_CONFIGS.keys())}")
print(f"  Seeds:      {SEEDS}")
print(f"  Total planned runs: {n_total}")

## Section 2 · Kvasir-SEG Dataset

### Why Kvasir-SEG for AEIoU validation?

Kvasir-SEG (Jha et al., 2020) contains **1 000 colonoscopy images** of colorectal
polyps annotated with pixel-level segmentation masks. Key properties that make it
ideal for testing AEIoU:

| Property | Value |
|---|---|
| Images | 1 000 |
| Classes | 1 (polyp) |
| Annotation type | Segmentation mask → converted to bbox |
| Avg polyp area | ~20–60% of image |
| Shape variability | High — round, elongated, flat, pedunculated |
| Boundary regularity | Low — irregular, lobular, concave edges |

### Amorphousness argument
A gastroenterologist annotating a polyp draws a rough contour around a blob with
no fixed shape. The resulting bounding box extent is **annotation-dependent**, not
geometry-determined. Two annotators will produce different box sizes for the same
polyp. This is exactly the regime where λ < 1 in AEIoU should help:
setting λ=0.3 tells the loss "the box center is reliable, but the width/height is
only 30% trustworthy."

### Data preparation pipeline
```
kvasir-seg.zip  →  images/ + masks/
                          ↓  (Cell 8: cv2.findContours + boundingRect)
                     YOLO labels: 0 cx cy w h  (normalised)
                          ↓  (80/20 split)
                     train/  +  valid/
                          ↓  (Cell 9: Gaussian jitter)
                     valid_low/ (σ=0.02)  +  valid_high/ (σ=0.08)
```

**What to look for:** Cell 10 should show 800 train images and 200 val images in each
of the three splits. If counts differ, the split or download step failed.


In [ ]:
# --- WandB setup — proper login with project configuration
# WANDB_API_KEY must be set as a Colab secret (Secrets panel, left sidebar).
# If not set, WandB is disabled so training still runs without hanging.
import os, wandb

# Project name used for all runs in this notebook
WANDB_PROJECT = "amorphous-yolo-kvasir"

# Try to read API key from Colab secrets first (most secure), then fall back
# to an existing environment variable set before this cell ran.
try:
    from google.colab import userdata
    api_key = userdata.get("WANDB_API_KEY")
    if api_key:
        os.environ["WANDB_API_KEY"] = api_key
        print("WandB API key loaded from Colab secrets.")
except Exception:
    pass   # not in Colab or secret not set

if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=False)
    print(f"WandB logged in. Project: {WANDB_PROJECT}")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not found — WandB disabled.")
    print("To enable: add WANDB_API_KEY in Colab Secrets (key icon, left sidebar).")


### Google Drive: Mount, Restore & Persist

Results are written to Drive **after every training run** so a Colab timeout
only loses the run currently in progress. On session restart, completed runs
are restored from Drive so the notebook resumes exactly where it left off.

**What to look for:** The restore step should print a list of run names copied
back from Drive. These runs will show `[SKIP]` in the training cells below,
meaning no computation is wasted repeating them.

**Setup:** Drive will be mounted automatically below. If it fails (e.g. running
locally), training still works — Drive sync is silently skipped.


In [ ]:
# --- Google Drive: mount + restore completed runs from previous sessions
import shutil

def mount_drive():
    # Mount Google Drive and set DRIVE_AVAILABLE = True if successful.
    global DRIVE_AVAILABLE
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f"Drive mounted. Backup dir: {DRIVE_EXPERIMENTS}")
    except Exception as e:
        print(f"Drive not available ({e}). Running without Drive persistence.")
        DRIVE_AVAILABLE = False
    return DRIVE_AVAILABLE


def restore_from_drive():
    # Copy completed runs from Drive back to local EXPERIMENTS dir.
    # Called once at session start so skip-on-existing logic works correctly
    # even after a Colab timeout/restart.
    if not DRIVE_AVAILABLE:
        return
    if not DRIVE_EXPERIMENTS.exists():
        print("No Drive backup found yet — starting fresh.")
        return
    restored = 0
    for drive_run in sorted(DRIVE_EXPERIMENTS.iterdir()):
        if not drive_run.is_dir():
            continue
        local_run = EXPERIMENTS / drive_run.name
        # Only restore runs that completed (have results.csv) and aren't already local
        if (drive_run / "results.csv").exists() and not (local_run / "results.csv").exists():
            shutil.copytree(str(drive_run), str(local_run), dirs_exist_ok=True)
            restored += 1
            print(f"  [RESTORE] {drive_run.name}")
    if restored == 0:
        print("Nothing to restore — local experiments are up to date.")
    else:
        print(f"Restored {restored} completed run(s) from Drive.")


mount_drive()
restore_from_drive()


In [ ]:
# --- Download Kvasir-SEG (idempotent)
# Source: Simula Research Lab official server — no API key required.
# File size: ~44 MB zip. Extracts to datasets/kvasir-seg/images/ and masks/.
import zipfile, urllib.request, shutil

kvasir_zip  = PROJECT_DIR / "datasets" / "kvasir-seg.zip"
images_dir  = DATASET_ROOT / "images"

if images_dir.exists() and len(list(images_dir.glob("*.jpg"))) > 900:
    print(f"Kvasir-SEG already present ({len(list(images_dir.glob('*.jpg')))} images) — skipping download.")
else:
    print("Downloading Kvasir-SEG (~44 MB)...")
    (PROJECT_DIR / "datasets").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://datasets.simula.no/downloads/kvasir-seg.zip",
        kvasir_zip,
    )
    print(f"  Downloaded → {kvasir_zip}")

    # Extract: creates kvasir-seg/ folder inside datasets/
    with zipfile.ZipFile(kvasir_zip, "r") as z:
        z.extractall(PROJECT_DIR / "datasets")
    print(f"  Extracted  → {DATASET_ROOT}")

    # Verify extracted structure
    n_imgs  = len(list((DATASET_ROOT / "images").glob("*.jpg")))
    n_masks = len(list((DATASET_ROOT / "masks").glob("*.jpg")))
    print(f"  Images: {n_imgs}  |  Masks: {n_masks}")
    assert n_imgs == n_masks == 1000, f"Expected 1000 pairs, got {n_imgs}/{n_masks}"
    print("Download complete.")


In [ ]:
# --- Convert segmentation masks -> YOLO bbox labels + 80/20 train/val split
# Kvasir-SEG provides PNG segmentation masks (white=polyp, black=background).
# We find the bounding box of all non-zero pixels using cv2.findContours,
# then write a YOLO label file: "0 cx cy w h" (all normalised to [0,1]).
#
# Each image has at most one polyp region (single-class dataset), so each
# label file contains exactly one line.
import cv2
import numpy as np
import random
import shutil

TRAIN_DIR  = DATASET_ROOT / "train"
VALID_DIR  = DATASET_ROOT / "valid"

def _mask_to_yolo_bbox(mask_path):
    # Load a Kvasir-SEG mask and return (cx, cy, w, h) normalised to [0,1].
    # Returns None if the mask is blank (no polyp found).
    # Read mask as grayscale; Kvasir masks are 3-channel JPEGs with white polyp
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    # Threshold to binary: pixel > 127 → foreground
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find contours of the polyp region(s)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None  # Blank mask — skip this image

    # Union bounding rect across all contours (handles multi-fragment masks)
    all_pts = np.vstack(contours)  # shape: [N_pts, 1, 2]
    x, y, w, h = cv2.boundingRect(all_pts)

    # Normalise by image dimensions (H, W)
    H, W = mask.shape
    cx = (x + w / 2) / W
    cy = (y + h / 2) / H
    bw = w / W
    bh = h / H

    # Clamp to [0, 1] for safety (rounding at image edges)
    cx, cy, bw, bh = [float(np.clip(v, 0.0, 1.0)) for v in [cx, cy, bw, bh]]
    return cx, cy, bw, bh


if (TRAIN_DIR / "images").exists() and len(list((TRAIN_DIR/"images").glob("*.jpg"))) >= 800:
    print("Train/val split already exists — skipping conversion.")
else:
    print("Converting masks → YOLO labels and creating train/val split...")
    for split_dir in [TRAIN_DIR, VALID_DIR]:
        (split_dir / "images").mkdir(parents=True, exist_ok=True)
        (split_dir / "labels").mkdir(parents=True, exist_ok=True)

    # Collect all image stems and shuffle for reproducible split
    all_imgs = sorted((DATASET_ROOT / "images").glob("*.jpg"))
    random.seed(42)
    random.shuffle(all_imgs)

    # 80% train, 20% val
    split_idx = int(0.8 * len(all_imgs))
    splits = {"train": all_imgs[:split_idx], "valid": all_imgs[split_idx:]}

    skipped = 0
    for split_name, img_list in splits.items():
        out_img_dir = DATASET_ROOT / split_name / "images"
        out_lbl_dir = DATASET_ROOT / split_name / "labels"
        for img_path in img_list:
            stem = img_path.stem
            # Kvasir masks share the same filename as images (jpg)
            mask_path = DATASET_ROOT / "masks" / f"{stem}.jpg"
            bbox = _mask_to_yolo_bbox(mask_path)
            if bbox is None:
                skipped += 1
                continue
            cx, cy, bw, bh = bbox
            # Copy image (hard copy so paths are self-contained)
            shutil.copy2(img_path, out_img_dir / img_path.name)
            # Write YOLO label: class_id cx cy w h
            (out_lbl_dir / f"{stem}.txt").write_text(
                f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}
"
            )

    print(f"  Conversion complete. Skipped {skipped} blank masks.")
    print(f"  Train: {len(list((TRAIN_DIR/'images').glob('*.jpg')))} images")
    print(f"  Valid: {len(list((VALID_DIR/'images').glob('*.jpg')))} images")


In [ ]:
# --- Build noise-perturbed validation splits (idempotent)
# Applies Gaussian jitter to YOLO bbox coordinates in valid/labels/ to simulate
# annotation noise. Unlike DUO (polygon format), Kvasir labels are standard
# axis-aligned YOLO boxes so jitter is applied directly to (cx, cy, w, h).
#
# sigma_low  = 0.02: ≈2% of image dimension → mild annotation imprecision
# sigma_high = 0.08: ≈8% of image dimension → severe noise, stress-test
import numpy as np
import shutil
from pathlib import Path

SIGMA_LOW  = 0.02
SIGMA_HIGH = 0.08

def _jitter_label_file(src_lbl, dst_lbl, sigma, rng):
    """Perturb all bbox coords in a YOLO label file by N(0, sigma) and clip to [0,1]."""
    lines = Path(src_lbl).read_text().strip().split("\n")
    out = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        cls_id = parts[0]
        cx, cy, w, h = [float(v) for v in parts[1:5]]
        # Add independent Gaussian noise to each coordinate
        # Clamp: ensures boxes stay inside image boundaries
        cx = float(np.clip(cx + rng.normal(0, sigma), 0.0, 1.0))
        cy = float(np.clip(cy + rng.normal(0, sigma), 0.0, 1.0))
        w  = float(np.clip(w  + rng.normal(0, sigma), 0.01, 1.0))  # min 1% width
        h  = float(np.clip(h  + rng.normal(0, sigma), 0.01, 1.0))  # min 1% height
        out.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    Path(dst_lbl).write_text("\n".join(out) + "\n")


def build_kvasir_noise_splits(root: Path, seed: int = 42):
    """Build valid_low/ and valid_high/ from valid/. Idempotent."""
    src_img_dir = root / "valid" / "images"
    src_lbl_dir = root / "valid" / "labels"
    rng = np.random.default_rng(seed)

    for split_name, sigma in [("valid_low", SIGMA_LOW), ("valid_high", SIGMA_HIGH)]:
        dst_img_dir = root / split_name / "images"
        dst_lbl_dir = root / split_name / "labels"

        if dst_img_dir.exists() and len(list(dst_img_dir.glob("*.jpg"))) >= 200:
            print(f"  {split_name}: already exists — skipping.")
            continue

        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path in sorted(src_img_dir.glob("*.jpg")):
            stem = img_path.stem
            # Images are identical — only labels change
            # Use symlink where possible; fall back to copy on Windows
            dst_img = dst_img_dir / img_path.name
            if not dst_img.exists():
                try:
                    dst_img.symlink_to(img_path)
                except (OSError, NotImplementedError):
                    shutil.copy2(img_path, dst_img)

            src_lbl = src_lbl_dir / f"{stem}.txt"
            dst_lbl = dst_lbl_dir / f"{stem}.txt"
            if src_lbl.exists():
                _jitter_label_file(src_lbl, dst_lbl, sigma, rng)

        n = len(list(dst_img_dir.glob("*.jpg")))
        print(f"  {split_name}: {n} images (σ={sigma})")


build_kvasir_noise_splits(DATASET_ROOT)
print("Noise splits ready.")


In [ ]:
# --- Verify all three split configs exist and have correct image counts
import yaml

print(f"{'Split':<10} {'YAML':<40} {'Val images':>12}")
print("-" * 65)
for split_name, cfg_path in SPLIT_CONFIGS.items():
    assert cfg_path.exists(), f"Missing yaml: {cfg_path}"
    cfg = yaml.safe_load(cfg_path.read_text())
    val_dir = DATASET_ROOT / cfg["val"]
    n_imgs = len(list(val_dir.glob("*.jpg")))
    status = "✓" if n_imgs >= 200 else "✗ CHECK"
    print(f"{split_name:<10} {str(cfg_path.name):<40} {n_imgs:>10}  {status}")

# Training split (same for all configs)
n_train = len(list((DATASET_ROOT / "train" / "images").glob("*.jpg")))
print(f"\nTrain images (shared): {n_train}")
assert n_train >= 800, f"Expected 800 train images, found {n_train}"
print("All splits verified.")


### Kvasir-SEG Sample Images

The following cell displays 5 random training images with ground-truth bounding boxes
overlaid in green. Labels show normalised box dimensions.

**What to look for:**
- Polyps should vary in size from small (<10% of image area) to large (>50%)
- Some polyps are near-circular; others are elongated or irregular blobs
- No two polyps look the same — this confirms the amorphous-object assumption
- The GT box should tightly contain the polyp with minimal slack

If boxes look misaligned, the mask→bbox conversion in Cell 8 may have a coordinate error.


In [ ]:
# --- Visualise 5 random training images with GT bounding boxes
# This is a qualitative sanity check: do the labels look correct?
import cv2, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def _draw_yolo_bbox(ax, label_path, img_w, img_h, color="lime", lw=2):
    """Draw YOLO-format bbox(es) from a label file onto a matplotlib axis."""
    if not Path(label_path).exists():
        return
    for line in Path(label_path).read_text().strip().split("\n"):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        # Convert normalised (cx,cy,w,h) → pixel (x1,y1,w,h) for matplotlib
        x1 = (cx - bw / 2) * img_w
        y1 = (cy - bh / 2) * img_h
        rect = patches.Rectangle(
            (x1, y1), bw * img_w, bh * img_h,
            linewidth=lw, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"polyp  w={bw:.2f} h={bh:.2f}",
                color=color, fontsize=7, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.5, pad=1))


train_imgs = sorted((DATASET_ROOT / "train" / "images").glob("*.jpg"))
sample = random.sample(train_imgs, 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Kvasir-SEG — 5 random training images with GT bounding boxes",
             fontsize=13, fontweight="bold")

for ax, img_path in zip(axes, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    ax.imshow(img)
    lbl_path = DATASET_ROOT / "train" / "labels" / f"{img_path.stem}.txt"
    _draw_yolo_bbox(ax, lbl_path, W, H)
    ax.set_title(img_path.stem[:20], fontsize=8)
    ax.axis("off")

plt.tight_layout()
save_path = ANALYSIS_DIR / "00_sample_images.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/00_sample_images.png
plt.show()
print(f"Saved → {save_path}")


**Figure 0 · Sample Kvasir-SEG Training Images**

*x-axis / y-axis:* pixel coordinates · *Green boxes:* ground-truth YOLO bounding boxes
*Labels:* show normalised width and height of each box

**Expected pattern:** Boxes of varying sizes and aspect ratios — small flat lesions
alongside large pedunculated polyps. Irregular polyp shapes that extend to one side of
the box are normal and confirm the amorphous assumption.
If all boxes appear the same size/shape, the mask conversion may have a bug.


## Section 3 · Theory: IoU Loss Family & AEIoU

### The IoU Loss Family — Complete Comparison

All losses share $1 - \text{IoU}$ as the base overlap term. They differ in what
auxiliary penalties they add and how they normalise them.

$$L_{\text{IoU}}   = 1 - \text{IoU}$$
$$L_{\text{GIoU}}  = 1 - \text{IoU} + \frac{|C \setminus (A \cup B)|}{|C|}$$
$$L_{\text{DIoU}}  = 1 - \text{IoU} + \frac{\rho^2(b,b^{gt})}{c^2}$$
$$L_{\text{CIoU}}  = 1 - \text{IoU} + \frac{\rho^2}{c^2} + \alpha v$$
$$L_{\text{EIoU}}  = 1 - \text{IoU} + \frac{\rho^2}{c^2} + \frac{(p_w-t_w)^2}{c_w^2} + \frac{(p_h-t_h)^2}{c_h^2}$$
$$L_{\text{ECIoU}} = 1 - \text{IoU} + \frac{\rho^2}{c^2} + \frac{(p_w-t_w)^2}{\max(p_w,t_w)^2} + \frac{(p_h-t_h)^2}{\max(p_h,t_h)^2}$$
$$L_{\text{AEIoU}} = 1 - \text{IoU} + \frac{\rho^2}{c^2} + \lambda\!\left(\frac{(p_w-t_w)^2}{t_w^2} + \frac{(p_h-t_h)^2}{t_h^2}\right)$$

where $\rho^2$ = squared center distance, $c^2$ = enclosing box diagonal$^2$,
$v = \frac{4}{\pi^2}(\arctan\frac{t_w}{t_h} - \arctan\frac{p_w}{p_h})^2$ (CIoU aspect ratio).

### Size-Term Normaliser Comparison — The Core Question

| Loss | Size term | Normaliser | Scales with |
|---|---|---|---|
| IoU | None | — | Pure overlap |
| GIoU | Enclosing box fill | $\|C\|$ | Scene context |
| DIoU | Center only | — | No size penalty |
| CIoU | Aspect ratio $v$ | Implicit | w/h ratio, not absolute size |
| **EIoU** | $(p_w-t_w)^2 + (p_h-t_h)^2$ | **Enclosing dims** $c_w^2, c_h^2$ | Scene clutter |
| **ECIoU** | $(p_w-t_w)^2 + (p_h-t_h)^2$ | **max(pred, target)** $^2$ | Larger box |
| **AEIoU** | $(p_w-t_w)^2 + (p_h-t_h)^2$ | **Target dims** $t_w^2, t_h^2$ × $\lambda$ | Label only |

### Three normalisation philosophies

**EIoU (enclosing box):** A 10 px width error is penalised less if the enclosing
box is 200 px wide. Sensible for rigid objects in cluttered scenes.

**ECIoU (max of pred/target):** The penalty scales with the larger of the two boxes.
More stable than EIoU when the enclosing box is much larger than both boxes.

**AEIoU (target dims + λ):** A 10 px error on a 20 px polyp = 25% of target$^2$.
The same error on a 200 px polyp = 0.25%. This is *label-relative*, not context-relative.
The λ parameter additionally down-weights the entire size term to account for
annotation noise in amorphous-object labels.

### The λ-rigidity argument for polyps

A gastroenterologist annotating a polyp draws a rough contour. The resulting
bounding box extent depends on where they stopped drawing, not the polyp's true geometry.
Two clinicians produce different box widths/heights for the same polyp.

Setting λ < 1 says: *"I trust the center coordinates more than the extent."*
- λ = 0.1: near-pure center-alignment loss — ignore width/height error
- λ = 0.3: down-weight size penalty 70% — moderate scepticism about extent labels
- λ = 1.0: full size penalty — same strength as EIoU (but different normaliser)

**Prediction:** The optimal λ for Kvasir-SEG will be 0.2–0.4, matching the DUO
dataset result, suggesting a domain-agnostic optimal λ for amorphous objects.


In [ ]:
# --- Numerical comparison: all 7 losses on 3 synthetic scenarios
# Unit-test style: verify losses respond correctly to canonical configs.
import torch
from src.losses import IoULoss, GIoULoss, DIoULoss, CIoULoss, EIoULoss, ECIoULoss, AEIoULoss

baseline_fns = {
    "IoU": IoULoss(reduction="none"),   "GIoU": GIoULoss(reduction="none"),
    "DIoU": DIoULoss(reduction="none"), "CIoU": CIoULoss(reduction="none"),
    "EIoU": EIoULoss(reduction="none"), "ECIoU": ECIoULoss(reduction="none"),
}
aeiou_fns = {f"AEIoU lam={r}": AEIoULoss(rigidity=r, reduction="none")
             for r in [0.1, 0.3, 0.5, 1.0]}
all_fns = {**baseline_fns, **aeiou_fns}

scenarios = {
    "Perfect match":   (torch.tensor([[10.,10.,50.,50.]]), torch.tensor([[10.,10.,50.,50.]])),
    "Partial overlap": (torch.tensor([[10.,10.,40.,40.]]), torch.tensor([[20.,20.,50.,50.]])),
    "No overlap":      (torch.tensor([[10.,10.,30.,30.]]), torch.tensor([[40.,40.,60.,60.]])),
}

import pandas as pd
rows = []
for scenario_name, (pred, target) in scenarios.items():
    row = {"Scenario": scenario_name}
    for fn_name, fn in all_fns.items():
        row[fn_name] = fn(pred, target).item()
    rows.append(row)

df_theory = pd.DataFrame(rows).set_index("Scenario")
print(df_theory.round(4).to_string())

print("\nExpected: loss decreases monotonically from 'No overlap' -> 'Perfect match'.")
print("AEIoU(lam=0.1) should be lowest on 'Partial overlap' (shape penalty suppressed).")
print("ECIoU should sit between EIoU and AEIoU(lam=1.0) on 'Partial overlap'.")


**Table · Numerical Loss Comparison (All 7 Losses)**

*Rows:* three canonical prediction/target configurations
*Columns:* all 6 baselines + AEIoU at four representative λ values

**Expected pattern:**
- All losses = 0.0 on "Perfect match", highest on "No overlap"
- IoU and GIoU have no explicit size penalty — they only see overlap
- DIoU adds center distance but no size term — same as IoU when centers coincide
- CIoU's aspect-ratio term $v$ is non-zero when pred/target have different shapes
- EIoU, ECIoU, AEIoU(λ=1.0) differ only in normaliser — values should be close but not equal
- AEIoU(λ=0.1) gives the lowest loss on "Partial overlap" — size penalty nearly zero


In [ ]:
# --- Loss sensitivity sweep: slide from no-overlap to perfect overlap
# Constructs a 1D sweep where the predicted box moves from no overlap (left)
# to perfect overlap (right) with the target. Plots loss value vs IoU.
# This reveals the gradient landscape each loss provides to the optimiser.
import matplotlib.pyplot as plt
import torch
import numpy as np
from src.losses import IoULoss, GIoULoss, DIoULoss, CIoULoss, EIoULoss, ECIoULoss, AEIoULoss

target = torch.tensor([[40., 40., 80., 80.]])  # Fixed target: 40x40 px box

# Sweep: predicted box x-offset from -40 (no overlap) to 0 (perfect overlap)
offsets = np.linspace(-40, 0, 200)
sweep_fns = {
    "IoU":  IoULoss(reduction="none"),  "GIoU": GIoULoss(reduction="none"),
    "DIoU": DIoULoss(reduction="none"), "CIoU": CIoULoss(reduction="none"),
    "EIoU": EIoULoss(reduction="none"), "ECIoU": ECIoULoss(reduction="none"),
}
aeiou_sweep = {f"AEIoU lam={r}": AEIoULoss(rigidity=r, reduction="none")
               for r in [0.1, 0.3, 0.5, 1.0]}
all_sweep = {**sweep_fns, **aeiou_sweep}

ious = []
sweep_vals = {k: [] for k in all_sweep}

for dx in offsets:
    pred = torch.tensor([[dx + 40., 40., dx + 80., 80.]]).clamp(min=0)
    # Compute IoU for x-axis
    inter_x = max(0, min(pred[0,2].item(), target[0,2].item()) - max(pred[0,0].item(), target[0,0].item()))
    inter_y = max(0, min(pred[0,3].item(), target[0,3].item()) - max(pred[0,1].item(), target[0,1].item()))
    inter = inter_x * inter_y
    area_p = (pred[0,2]-pred[0,0]) * (pred[0,3]-pred[0,1])
    area_t = (target[0,2]-target[0,0]) * (target[0,3]-target[0,1])
    iou = inter / (float(area_p) + float(area_t) - inter + 1e-7)
    ious.append(float(iou))
    for k, fn in all_sweep.items():
        sweep_vals[k].append(float(fn(pred, target)))

# Plot baselines as solid lines, AEIoU as dashed
fig, ax = plt.subplots(figsize=(12, 6))
base_colors = {"IoU":"#888","GIoU":"#BC6C25","DIoU":"#606C38","CIoU":"#DDA15E","EIoU":"#E63946","ECIoU":"#9B2226"}
for k in sweep_fns:
    ax.plot(ious, sweep_vals[k], color=base_colors[k], lw=2, label=k)
aeiou_colors = {"AEIoU lam=0.1":"#023E8A","AEIoU lam=0.3":"#00B4D8","AEIoU lam=0.5":"#90E0EF","AEIoU lam=1.0":"#6A4C93"}
for k in aeiou_sweep:
    ax.plot(ious, sweep_vals[k], color=aeiou_colors[k], lw=1.8, linestyle="--", label=k)

ax.set_xlabel("IoU with target box", fontsize=12)
ax.set_ylabel("Loss value", fontsize=12)
ax.set_title("Loss sensitivity: no-overlap -> perfect overlap (all 7 loss families)",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.invert_xaxis()  # left=no overlap (IoU=0), right=perfect (IoU=1)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "01_loss_sensitivity.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/01_loss_sensitivity.png
plt.show()
print(f"Saved → {save_path}")


**Figure 1 · Loss Sensitivity Sweep (All Loss Families)**

*x-axis:* IoU between predicted and target box (right = perfect overlap, left = no overlap)
*y-axis:* scalar loss value · *Solid lines:* baselines · *Dashed lines:* AEIoU variants

**Expected pattern:**
- All curves converge to 0 at IoU = 1.0 (rightmost point)
- IoU and GIoU have no explicit size term — flattest curves
- DIoU adds center distance but is identical to IoU when centers are aligned
- CIoU, EIoU, ECIoU add size penalties — highest curves at low IoU
- AEIoU(lam=0.1) has the smallest loss at IoU=0 (size penalty nearly zero)
- EIoU, ECIoU, and AEIoU(lam=1.0) should cluster together but not coincide
  (different normalisers = different gradient landscapes)


## Section 4 · Monkey-Patch Infrastructure

### Why monkey-patch instead of editing Ultralytics source?

Ultralytics is a versioned pip package. Editing its source would:
1. Break on `pip install --upgrade ultralytics`
2. Make the experiment hard to reproduce on a fresh Colab runtime
3. Require maintaining a fork

Instead, we replace `BboxLoss.forward` at runtime with a closure that:
- Calls our custom loss function for the **IoU term**
- Copies the **DFL term** verbatim from Ultralytics 8.4.9
- Is fully reversible via `restore_loss()` — critical between runs

The patch is applied immediately before `model.train()` and removed in a
`try/finally` block so it is guaranteed to be restored even if training crashes.

**What to look for in Cell 18:** Running `patch_loss(EIoULoss(...))` then
`restore_loss()` should leave `BboxLoss.forward` identical to its original form
(same function object reference). The print at the end confirms this.


In [ ]:
# --- Full monkey-patch implementation (verbatim pattern from notebook 02)
# BboxLoss.forward signature in ultralytics 8.4.9:
#   forward(self, pred_dist, pred_bboxes, anchor_points,
#           target_bboxes, target_scores, target_scores_sum, fg_mask, imgsz, stride)
import types
import torch
import torch.nn.functional as F
import ultralytics.utils.loss as loss_mod

# Save the original forward method ONCE at import time.
# This is the reference we restore after each training run.
_ORIGINAL_BBOX_FORWARD = loss_mod.BboxLoss.forward


def _make_bbox_forward(loss_fn_instance):
    """
    Factory: returns a bound-method replacement for BboxLoss.forward that
    uses loss_fn_instance for the IoU term and keeps DFL unchanged.

    loss_fn_instance must support reduction='none' and return shape [N].
    """
    def bbox_loss_forward(
        self,
        pred_dist, pred_bboxes, anchor_points,
        target_bboxes, target_scores, target_scores_sum,
        fg_mask, imgsz, stride,
    ):
        # ── IoU term ──────────────────────────────────────────────────────────
        # target_scores: [B, A, C] — score per anchor; sum over classes gives weight
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)  # shape: [N, 1]

        # Apply our custom loss to foreground anchors only (fg_mask selects them)
        # pred_bboxes and target_bboxes are in xyxy format, normalised to image size
        per_box = loss_fn_instance(
            pred_bboxes[fg_mask],    # [N, 4] xyxy predicted
            target_bboxes[fg_mask],  # [N, 4] xyxy ground truth
        )  # → [N] per-box loss values (reduction='none')

        # Weighted average over foreground anchors, normalised by total score sum
        loss_iou = (per_box.unsqueeze(-1) * weight).sum() / target_scores_sum

        # ── DFL term (copied verbatim from ultralytics 8.4.9) ─────────────────
        # DFL (Distribution Focal Loss) operates on the predicted distance
        # distribution; it is independent of the IoU metric choice.
        if self.dfl_loss:
            target_ltrb = loss_mod.bbox2dist(
                anchor_points, target_bboxes, self.dfl_loss.reg_max - 1
            )
            loss_dfl = self.dfl_loss(
                pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
                target_ltrb[fg_mask],
            ) * weight
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            target_ltrb = loss_mod.bbox2dist(anchor_points, target_bboxes)
            target_ltrb = target_ltrb * stride
            target_ltrb[..., 0::2] /= imgsz[1]
            target_ltrb[..., 1::2] /= imgsz[0]
            pred_dist_s = pred_dist * stride
            pred_dist_s[..., 0::2] /= imgsz[1]
            pred_dist_s[..., 1::2] /= imgsz[0]
            loss_dfl = (
                F.l1_loss(pred_dist_s[fg_mask], target_ltrb[fg_mask],
                          reduction="none").mean(-1, keepdim=True) * weight
            )
            loss_dfl = loss_dfl.sum() / target_scores_sum

        return loss_iou, loss_dfl

    return bbox_loss_forward


def patch_loss(loss_fn_instance):
    """Apply custom loss patch. Always call restore_loss() before patching again."""
    loss_mod.BboxLoss.forward = _make_bbox_forward(loss_fn_instance)
    print(f"  [PATCH] BboxLoss.forward → {type(loss_fn_instance).__name__}"
          + (f"(λ={loss_fn_instance.rigidity})" if hasattr(loss_fn_instance, "rigidity") else ""))


def restore_loss():
    """Restore the original Ultralytics CIoU-based BboxLoss.forward."""
    loss_mod.BboxLoss.forward = _ORIGINAL_BBOX_FORWARD


print("Patch infrastructure ready.")
print(f"  Original forward saved: {_ORIGINAL_BBOX_FORWARD}")


In [ ]:
# --- Verify patch round-trip
from src.losses import EIoULoss

# Step 1: apply patch
patch_loss(EIoULoss(reduction="none"))
patched_forward = loss_mod.BboxLoss.forward
print(f"After patch:   BboxLoss.forward is <function bbox_loss_forward>: "
      f"{'bbox_loss_forward' in str(patched_forward)}")

# Step 2: restore
restore_loss()
restored_forward = loss_mod.BboxLoss.forward
print(f"After restore: BboxLoss.forward is original: "
      f"{restored_forward is _ORIGINAL_BBOX_FORWARD}")

assert restored_forward is _ORIGINAL_BBOX_FORWARD, "restore_loss() failed!"
print("Patch round-trip verified.")


## Section 5 · Loss Registry

All **6 standard IoU-family losses** and **10 AEIoU variants** are instantiated here.
Each loss is monkey-patched into Ultralytics one at a time during training.
Comparing against all baselines (not just EIoU) ensures the paper's claims are
robust: AEIoU must beat the **entire field**, not just one competitor.

| Baseline | Publication | Key innovation |
|---|---|---|
| IoU | Yu et al., 2016 | Pure overlap, no auxiliary penalties |
| GIoU | Rezatofighi et al., 2019 | Enclosing area fill penalty |
| DIoU | Zheng et al., 2020 | Center-distance penalty |
| CIoU | Zheng et al., 2020 | + aspect-ratio consistency penalty |
| EIoU | Zheng et al., 2021 | Decoupled width/height penalty, enclosing-box normalised |
| ECIoU | Zhang et al., 2023 | max(pred,target) normalised |
| **AEIoU** | **Ours** | Target-normalised + λ-rigidity |


In [ ]:
# --- Loss registry: all baselines + AEIoU grid
from src.losses import IoULoss, GIoULoss, DIoULoss, CIoULoss, EIoULoss, ECIoULoss, SIoULoss, WIoULoss, AEIoULoss

# All 6 standard baselines — reduction='none' required by the monkey-patch
BASELINE_LOSS_REGISTRY = {
    "iou":   IoULoss(reduction="none"),
    "giou":  GIoULoss(reduction="none"),
    "diou":  DIoULoss(reduction="none"),
    "ciou":  CIoULoss(reduction="none"),
    "eiou":  EIoULoss(reduction="none"),
    "eciou": ECIoULoss(reduction="none"),
    "siou":  SIoULoss(reduction="none"),
    "wiou":  WIoULoss(reduction="none"),
}

# AEIoU grid: 10 instances, one per λ value
AEIOU_LOSS_REGISTRY = {
    f"aeiou_r{_fmt_r(r)}": AEIoULoss(rigidity=r, reduction="none")
    for r in AEIOU_RIGIDITIES
}

# Combined registry for easy iteration
ALL_LOSS_REGISTRY = {**BASELINE_LOSS_REGISTRY, **AEIOU_LOSS_REGISTRY}

print(f"Baselines ({len(BASELINE_LOSS_REGISTRY)}):")
for name, fn in BASELINE_LOSS_REGISTRY.items():
    print(f"  {name}: {type(fn).__name__}")

print(f"\nAEIoU grid ({len(AEIOU_LOSS_REGISTRY)}):")
for name, fn in AEIOU_LOSS_REGISTRY.items():
    print(f"  {name}: rigidity={fn.rigidity}")

n_total = len(ALL_LOSS_REGISTRY) * len(SPLIT_CONFIGS) * len(SEEDS)
print(f"\nTotal: {len(ALL_LOSS_REGISTRY)} losses x {len(SPLIT_CONFIGS)} splits "
      f"x {len(SEEDS)} seed(s) = {n_total} runs")

## Section 6 · Training Infrastructure

### Run naming convention
```
kvasir_yolo26n_{loss_key}_{split}_e{epochs}
```
Examples:
- `kvasir_yolo26n_eiou_clean_e20`
- `kvasir_yolo26n_aeiou_r0p3_low_e20`
- `kvasir_yolo26n_aeiou_r1p0_high_e20`

`loss_key` always comes from the registry — never constructed ad-hoc.

### Idempotency
`run_training()` checks if `{run_dir}/results.csv` already exists.
If yes → `[SKIP]`. This means any interrupted session can be resumed by
re-running the training cells without duplicating completed runs.

### Manifest
`experiments_kvasir/manifest.json` is updated after **every** run (even failures).
Status field: `"running"` → `"complete"` or `"failed"`. This allows resuming
interrupted sessions and auditing which runs succeeded.

### GPU requirements
- T4 (16 GB): ~3–5 min/run × 33 runs = ~2.5 hrs total
- A100 (40 GB): ~1–2 min/run = ~1 hr total
- Recommended: rent an A100 or L4 for this notebook

**What to look for:** Each run's stdout should show `[START]` then Ultralytics'
training progress bar. `[SKIP]` means the run was already completed — safe to ignore.


In [ ]:
# --- Training functions: run_training + write_manifest_entry + sync_to_drive
# --- + make_epoch_checkpoint_callback (epoch-level Drive sync for mid-run resume)
import json, shutil
from pathlib import Path as _Path
from datetime import datetime
from ultralytics import YOLO

def _load_manifest():
    # Load manifest.json as a dict, or return empty dict if not present.
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {}


def write_manifest_entry(run_name, meta):
    # Atomically update manifest.json with the entry for run_name.
    manifest = _load_manifest()
    manifest[run_name] = meta
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))


def sync_to_drive(run_name):
    # Copy a single completed run directory from local storage to Google Drive.
    # Called in run_training()'s finally block so every run is persisted
    # immediately — a Colab timeout after this point loses nothing.
    if not DRIVE_AVAILABLE:
        return
    local_run = EXPERIMENTS / run_name
    drive_run = DRIVE_EXPERIMENTS / run_name
    try:
        shutil.copytree(str(local_run), str(drive_run), dirs_exist_ok=True)
        print(f"  [DRIVE] Synced {run_name}")
    except Exception as e:
        print(f"  [DRIVE] Sync failed for {run_name}: {e}")


def make_epoch_checkpoint_callback(run_name):
    # Returns an Ultralytics callback that copies last.pt to Drive after every epoch.
    # Ultralytics calls on_train_epoch_end(trainer) automatically each epoch.
    # trainer.save_dir points to the run directory (e.g. experiments_kvasir/run_name/).
    # yolo26n last.pt is ~20 MB — negligible Drive quota per epoch.
    # This enables mid-run resume: if Colab disconnects at epoch N, the next session
    # restores last.pt from Drive and calls model.train(resume=True) to continue.
    def _on_epoch_end(trainer):
        if not DRIVE_AVAILABLE:
            return
        last_pt = _Path(trainer.save_dir) / "weights" / "last.pt"
        if not last_pt.exists():
            return
        drive_weights = DRIVE_EXPERIMENTS / run_name / "weights"
        drive_weights.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(str(last_pt), str(drive_weights / "last.pt"))
        except Exception as e:
            # Non-fatal: training continues even if the epoch sync fails.
            print(f"  [DRIVE] Epoch checkpoint sync failed at epoch: {e}")
    return _on_epoch_end


def run_training(loss_name, loss_fn, split_name, yaml_path,
                 seed=42, epochs=None, imgsz=None, device=None):
    # Train one YOLO26n model. Returns the experiment run directory Path.
    # Supports three scenarios transparently:
    #   1. Fresh start  — no local or Drive checkpoint
    #   2. Resume       — Drive has last.pt from an interrupted run (no local results.csv)
    #   3. Skip         — local results.csv exists (run already completed)
    epochs = epochs if epochs is not None else EPOCHS
    imgsz  = imgsz  if imgsz  is not None else IMGSZ
    device = device if device is not None else DEVICE

    # Canonical run name — includes seed for multi-seed experiments
    run_name = f"kvasir_yolo26n_{loss_name}_{split_name}_s{seed}_e{epochs}"
    run_dir  = EXPERIMENTS / run_name

    # ── Skip if already completed ──────────────────────────────────────────────
    if (run_dir / "results.csv").exists():
        print(f"[SKIP] {run_name}")
        return run_dir

    # ── Check for a Drive checkpoint from an interrupted run ───────────────────
    # last.pt on Drive means a previous session started this run but did not finish.
    # We restore the checkpoint locally so Ultralytics can resume training.
    drive_last_pt = DRIVE_EXPERIMENTS / run_name / "weights" / "last.pt"
    resuming = DRIVE_AVAILABLE and drive_last_pt.exists()

    if resuming:
        print(f"\n{'='*70}")
        print(f"[RESUME] {run_name}")
        print(f"  Checkpoint found on Drive — resuming from last saved epoch")
        print(f"{'='*70}")
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "weights").mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(drive_last_pt), str(run_dir / "weights" / "last.pt"))
    else:
        print(f"\n{'='*70}")
        print(f"[START] {run_name}")
        print(f"  loss={loss_name}  split={split_name}  seed={seed}  epochs={epochs}")
        print(f"{'='*70}")

    # Record intent in manifest before training (status='running')
    # This allows detecting crashed runs that never reached status='complete'
    meta = {
        "loss":        loss_name,
        "split":       split_name,
        "seed":        seed,
        "epochs":      epochs,
        "rigidity":    float(getattr(loss_fn, "rigidity", -1) or -1),
        "run_dir":     str(run_dir),
        "weights":     str(run_dir / "weights" / "best.pt"),
        "results_csv": str(run_dir / "results.csv"),
        "timestamp":   datetime.now().isoformat(),
        "status":      "running",
        "resumed":     resuming,
    }
    write_manifest_entry(run_name, meta)

    t_start = time.time()
    try:
        # Name this run in WandB before training starts so it appears correctly
        # in the dashboard even if the session is interrupted mid-run.
        import os as _os
        _os.environ["WANDB_PROJECT"] = WANDB_PROJECT
        _os.environ["WANDB_NAME"]    = run_name
        _os.environ["WANDB_TAGS"]    = f"{loss_name},{split_name}"

        # Apply our custom loss to BboxLoss.forward
        patch_loss(loss_fn)

        if resuming:
            model = YOLO(str(run_dir / "weights" / "last.pt"))
            model.add_callback("on_train_epoch_end",
                               make_epoch_checkpoint_callback(run_name))
            results = model.train(resume=True)
        else:
            model = YOLO(MODEL_PT)
            model.add_callback("on_train_epoch_end",
                               make_epoch_checkpoint_callback(run_name))
            results = model.train(
                data=str(yaml_path),
                epochs=epochs,
                imgsz=imgsz,
                project=str(EXPERIMENTS),
                name=run_name,
                device=device,
                seed=seed,
                exist_ok=True,
            )

        # Snapshot Ultralytics metadata for offline analysis
        try:
            (run_dir / "run_meta.json").write_text(
                json.dumps(results.results_dict, indent=2)
            )
        except Exception as e:
            print(f"  [WARN] Could not write run_meta.json: {e}")

        meta["status"] = "complete"
        meta["elapsed_sec"] = round(time.time() - t_start, 1)

        # Explicitly finish the WandB run so metrics are fully uploaded
        # before we move to the next training run.
        try:
            import wandb as _wandb
            if _wandb.run is not None:
                _wandb.finish()
        except Exception:
            pass

    except Exception as e:
        print(f"  [ERROR] {run_name} failed: {e}")
        meta["status"] = "failed"
        meta["error"]  = str(e)
        raise

    finally:
        # CRITICAL: always restore — never leave Ultralytics in patched state
        restore_loss()
        write_manifest_entry(run_name, meta)
        # Sync this run to Drive immediately — protects against future timeouts.
        # A Colab crash after this line loses at most the next run, never this one.
        sync_to_drive(run_name)

    print(f"[DONE] {run_name}")
    return run_dir


print("run_training() ready.")
print(f"Manifest path: {MANIFEST_PATH}")


## Section 7 · All Baseline Training (IoU, GIoU, DIoU, CIoU, EIoU, ECIoU, SIoU, WIoU)

Training all 8 standard IoU-family losses as baselines. Each baseline is trained
on all 3 splits (clean, low-noise, high-noise) and all seeds.

| Loss | Innovation | Splits | Seeds | Runs |
|---|---|---|---|---|
| IoU | Pure overlap | 3 | per SEEDS | 3×N |
| GIoU | + enclosing area | 3 | per SEEDS | 3×N |
| DIoU | + center distance | 3 | per SEEDS | 3×N |
| CIoU | + aspect ratio | 3 | per SEEDS | 3×N |
| EIoU | + decoupled W/H (enclosing-normalised) | 3 | per SEEDS | 3×N |
| ECIoU | + decoupled W/H (max-normalised) | 3 | per SEEDS | 3×N |

**Expected runtime:** ~30-50 min for 18 runs on T4 (1 seed), ~90-150 min with 3 seeds.

In [ ]:
# --- Baseline training: all 6 standard losses × 3 splits × N seeds
# Re-running is safe — completed runs are skipped via [SKIP] logic.
for loss_name, loss_fn in BASELINE_LOSS_REGISTRY.items():
    for split_name, cfg_path in SPLIT_CONFIGS.items():
        for seed in SEEDS:
            run_training(
                loss_name=loss_name,
                loss_fn=loss_fn,
                split_name=split_name,
                yaml_path=cfg_path,
                seed=seed,
            )

restore_loss()  # Defensive restore
n_base = len(BASELINE_LOSS_REGISTRY) * len(SPLIT_CONFIGS) * len(SEEDS)
print(f"\nAll {n_base} baseline runs complete (or skipped).")


## Section 8 · AEIoU Rigidity Grid Search (30 runs)

We sweep λ from 0.1 to 1.0 in steps of 0.1 across all three splits:
- 10 λ values × 3 splits = **30 training runs**

### λ interpretation for polyp detection

| λ | Interpretation | Expected behaviour |
|---|---|---|
| 0.1 | Shape labels nearly ignored — pure center+overlap loss | High robustness, possibly lower clean mAP |
| 0.2–0.4 | Moderate shape trust — expected optimal range | Best clean mAP, good robustness |
| 0.5–0.7 | Stronger shape penalty | Converges to EIoU-like behaviour |
| 1.0 | Full shape penalty (different normaliser than EIoU) | ~= EIoU, confirms normaliser effect is small |

**Note:** `aeiou_r1p0` and `eiou` differ only in the size-term normaliser
(target dims vs enclosing dims). If their mAP is nearly identical, the normaliser
choice has minimal practical impact. If `aeiou_r1p0` noticeably underperforms `eiou`
on clean data, the target-dim normaliser may be too aggressive.

**Expected runtime:** ~90–150 minutes for all 30 runs on T4.
Grab a coffee. Or run overnight with a Colab Pro session.


In [ ]:
# --- AEIoU rigidity grid training (10 lambdas x 3 splits x N seeds)
# Outer loop: rigidity values (0.1 -> 1.0)
# Middle loop: splits (clean, low, high)
# Inner loop: seeds
# All runs are idempotent — re-run safely after interruption.

total = len(AEIOU_RIGIDITIES) * len(SPLIT_CONFIGS) * len(SEEDS)
done  = 0

for r in AEIOU_RIGIDITIES:
    loss_name = f"aeiou_r{_fmt_r(r)}"
    loss_fn   = AEIOU_LOSS_REGISTRY[loss_name]

    for split_name, cfg_path in SPLIT_CONFIGS.items():
        for seed in SEEDS:
            done += 1
            print(f"\n[{done}/{total}] lam={r}  split={split_name}  seed={seed}")
            run_training(
                loss_name=loss_name,
                loss_fn=loss_fn,
                split_name=split_name,
                yaml_path=cfg_path,
                seed=seed,
            )

restore_loss()  # Defensive final restore
print(f"\nAll {total} AEIoU grid runs complete (or skipped).")


## Section 9 · Results Collection

Ultralytics writes a `results.csv` file into each run directory after training.
This section collects all CSVs into a single flat DataFrame tagged with
`loss`, `split`, `seed`, and `rigidity` columns, then caches it to
`experiments_kvasir/all_results_combined.csv`.

**All downstream analysis cells load from this cache** — they never trigger
re-training. This means you can re-run Sections 10–13 on a fresh Colab session
(after uploading the cache CSV) without re-training any models.

**What to look for:** The combined DataFrame should have:
- One row per (run, epoch) combination
- No NaN in `metrics/mAP50-95(B)` at the final epoch
- `n_runs` matches the total planned runs


In [ ]:
# --- Load all results.csv files into a single flat DataFrame
import pandas as pd

CACHE_CSV = EXPERIMENTS / "all_results_combined.csv"

def load_all_results(force_rebuild=False):
    # Returns DataFrame with all training metrics tagged by loss, split, seed.
    if CACHE_CSV.exists() and not force_rebuild:
        print(f"Loading from cache: {CACHE_CSV}")
        return pd.read_csv(CACHE_CSV)

    print("Building combined results from individual CSVs...")
    dfs = []

    for loss_name in ALL_LOSS_KEYS:
        for split_name in SPLIT_CONFIGS:
            for seed in SEEDS:
                run_name = f"kvasir_yolo26n_{loss_name}_{split_name}_s{seed}_e{EPOCHS}"
                csv_path = EXPERIMENTS / run_name / "results.csv"
                if csv_path.exists():
                    df = pd.read_csv(csv_path)
                    df.columns = df.columns.str.strip()
                    df["run_name"] = run_name
                    df["loss"]     = loss_name
                    df["split"]    = split_name
                    df["seed"]     = seed
                    if "aeiou" in loss_name:
                        r_str = loss_name.split("_r")[1].replace("p",".")
                        df["rigidity"] = float(r_str)
                    else:
                        df["rigidity"] = -1.0
                    dfs.append(df)
                else:
                    print(f"  [MISSING] {csv_path}")

    if not dfs:
        raise RuntimeError("No results found. Run training cells first.")

    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(CACHE_CSV, index=False)
    print(f"Saved combined CSV: {CACHE_CSV}  ({len(combined)} rows)")
    return combined


df_all = load_all_results()
print(f"\nCombined DataFrame shape: {df_all.shape}")
print(f"Unique runs: {df_all['run_name'].nunique()}")
print(f"Losses found: {sorted(df_all['loss'].unique())}")
print(f"Seeds:  {sorted(df_all['seed'].unique())}")


## Section 10 · Summary Table

The master summary table: final-epoch mAP50 and mAP50-95 for every loss x split,
aggregated across seeds (mean +/- std when multiple seeds are present).
This is the central table for the paper — all analyses derive from it.


In [ ]:
# --- Build master pivot table with mean +/- std across seeds
import pandas as pd
import numpy as np

MAP50_COL  = "metrics/mAP50(B)"
MAP95_COL  = "metrics/mAP50-95(B)"

# Take the last epoch per run
df_final = (
    df_all.sort_values("epoch")
          .groupby("run_name")
          .last()
          .reset_index()
)

# Aggregate across seeds: mean and std per (loss, split)
agg_rows = []
for loss_name in ALL_LOSS_KEYS:
    for split in ["clean", "low", "high"]:
        sub = df_final[(df_final["loss"] == loss_name) & (df_final["split"] == split)]
        if sub.empty:
            continue
        row = {
            "loss": loss_name,
            "split": split,
            "label": LOSS_LABELS.get(loss_name, loss_name),
        }
        for col, tag in [(MAP95_COL, "map95"), (MAP50_COL, "map50")]:
            if col in sub.columns:
                row[f"{tag}_mean"] = sub[col].mean()
                row[f"{tag}_std"]  = sub[col].std() if len(sub) > 1 else 0.0
        agg_rows.append(row)

df_agg = pd.DataFrame(agg_rows)

# Pivot for display
pivot95 = df_agg.pivot_table(index="loss", columns="split", values="map95_mean")
pivot95 = pivot95[["clean", "low", "high"]]
pivot95["robust_ratio"] = pivot95["high"] / pivot95["clean"]
pivot95 = pivot95.sort_values("clean", ascending=False)

pivot50 = df_agg.pivot_table(index="loss", columns="split", values="map50_mean")
pivot50 = pivot50[["clean", "low", "high"]]

# Save
pivot95.to_csv(ANALYSIS_DIR / "summary_map95.csv")
pivot50.to_csv(ANALYSIS_DIR / "summary_map50.csv")

print("=== mAP50-95 (localisation quality) ===")
print(pivot95.round(4).to_string())
print(f"\nBest clean mAP50-95: {pivot95['clean'].idxmax()} = {pivot95['clean'].max():.4f}")
print(f"Best robust ratio:   {pivot95['robust_ratio'].idxmax()} = {pivot95['robust_ratio'].max():.4f}")

print("\n=== mAP50 (detection quality) ===")
print(pivot50.round(4).to_string())
print(f"\nBest clean mAP50: {pivot50['clean'].idxmax()} = {pivot50['clean'].max():.4f}")


## Section 11 · Core Analysis (Research-Level Figures)

15+ figures providing multi-angle evidence for the AEIoU hypothesis.
Each figure is saved to `experiments_kvasir/analysis/` as a high-DPI PNG.

| Figure | Key Question |
|---|---|
| Fig 2: mAP95 bar chart | Which loss achieves highest localisation quality? |
| Fig 2b: mAP50 bar chart | Which loss achieves best detection rate? |
| Fig 3: lambda-vs-mAP curve | Where in the lambda space does AEIoU peak? |
| Fig 4: lambda heatmap | Is optimal lambda stable across noise splits? |
| Fig 5: Learning curves | How do losses compare during training? |
| Fig 6: Convergence speed | Which loss reaches 90% mAP fastest? |
| Fig 7: Noise robustness | Which loss degrades least under annotation noise? |
| Fig 8: PR curves | Detection-recall trade-offs |
| Fig 9: AP@thresholds | Are gains consistent across IoU thresholds? |
| Fig 10: Training stability | Which loss produces smoothest training? |
| Fig 11: Box calibration | Do predicted box dimensions match GT? |
| Fig 12: Statistical tests | Are differences statistically significant? |
| Fig 13: Cross-dataset | Does optimal lambda match notebook 02 (DUO)? |
| Fig 14: Runtime overhead | Does AEIoU add computation cost? |
| Fig 15: IoU distribution | Per-prediction box quality |
| Fig 16: Size-stratified AP | Where does AEIoU help most? |
| Fig 17: Failure cases | Where does EIoU beat AEIoU? |


In [ ]:
# --- Fig 2: Final mAP50-95 bar chart — ALL losses grouped by split
import matplotlib.pyplot as plt
import numpy as np

splits = ["clean", "low", "high"]
split_colors = {"clean": "#4CAF50", "low": "#FFC107", "high": "#F44336"}
x_labels = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
display_labels = [LOSS_LABELS.get(l, l) for l in x_labels]
x_pos = np.arange(len(x_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(18, 6))
for i, split in enumerate(splits):
    vals = [pivot95.loc[l, split] if l in pivot95.index else 0 for l in x_labels]
    ax.bar(x_pos + i*width, vals, width=width,
           color=split_colors[split], alpha=0.85, label=split)

# Vertical line separating baselines from AEIoU
ax.axvline(x=len(BASELINE_LOSS_NAMES) - 0.5, color="black", linestyle=":", lw=1.5, alpha=0.4)
ax.text(len(BASELINE_LOSS_NAMES) - 0.3, ax.get_ylim()[1]*0.98, "baselines | AEIoU",
        fontsize=8, rotation=90, va="top")

ax.set_xticks(x_pos + width)
ax.set_xticklabels(display_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("mAP@[.5:.95]", fontsize=11)
ax.set_title("Final mAP50-95: All Losses Across 3 Noise Splits", fontsize=13, fontweight="bold")
ax.legend(title="Split", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "02_final_map95_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 2b: Final mAP50 bar chart — clinical detection metric
import matplotlib.pyplot as plt
import numpy as np

x_labels = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
display_labels = [LOSS_LABELS.get(l, l) for l in x_labels]
x_pos = np.arange(len(x_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(18, 6))
for i, split in enumerate(["clean", "low", "high"]):
    vals = [pivot50.loc[l, split] if l in pivot50.index else 0 for l in x_labels]
    color = {"clean": "#4CAF50", "low": "#FFC107", "high": "#F44336"}[split]
    ax.bar(x_pos + i*width, vals, width=width, color=color, alpha=0.85, label=split)

ax.axvline(x=len(BASELINE_LOSS_NAMES) - 0.5, color="black", linestyle=":", lw=1.5, alpha=0.4)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(display_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("mAP@0.5 (detection)", fontsize=11)
ax.set_title("Final mAP50: All Losses — Detection Quality (Clinical Metric)",
             fontsize=13, fontweight="bold")
ax.legend(title="Split", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "02b_final_map50_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


**Figures 2 & 2b: mAP50-95 vs mAP50**

mAP50-95 rewards tight, well-sized boxes (localisation). mAP50 rewards finding the
polyp at all (detection). For clinical use, mAP50 matters most — a gastroenterologist
needs to *see* the polyp. But for loss function research, mAP50-95 is the engineering
metric that isolates box quality from detection sensitivity.

**What to look for:** If AEIoU improves mAP50-95 but not mAP50, the gain is purely
in box tightness. If it improves both, the gain extends to detection — a stronger result.


In [ ]:
# --- Fig 3: lambda-vs-mAP continuous curve with baseline reference lines
# This is the key figure — shows exactly where in lambda space AEIoU peaks
# and whether it exceeds ALL baselines, not just EIoU.
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric_col, title_suffix in [
    (axes[0], "map95_mean", "mAP50-95 (localisation)"),
    (axes[1], "map50_mean", "mAP50 (detection)"),
]:
    for split, ls, marker in [("clean", "-", "o"), ("low", "--", "s"), ("high", ":", "^")]:
        # AEIoU curve: lambda vs mAP
        lambdas = []
        maps = []
        for r in AEIOU_RIGIDITIES:
            loss_name = f"aeiou_r{_fmt_r(r)}"
            row = df_agg[(df_agg["loss"] == loss_name) & (df_agg["split"] == split)]
            if not row.empty:
                lambdas.append(r)
                maps.append(row[metric_col].values[0])
        if lambdas:
            ax.plot(lambdas, maps, ls + marker, color="#00B4D8", lw=2,
                    markersize=5, label=f"AEIoU ({split})" if split == "clean" else f"_({split})")

        # Horizontal reference lines for each baseline
        for base_name in BASELINE_LOSS_NAMES:
            row = df_agg[(df_agg["loss"] == base_name) & (df_agg["split"] == split)]
            if not row.empty:
                val = row[metric_col].values[0]
                color = PALETTE.get(base_name, "#888")
                if split == "clean":
                    ax.axhline(y=val, color=color, linestyle="-.", lw=1.2, alpha=0.7,
                               label=LOSS_LABELS.get(base_name, base_name))

    ax.set_xlabel("AEIoU rigidity (lambda)", fontsize=11)
    ax.set_ylabel(title_suffix, fontsize=11)
    ax.set_title(f"AEIoU lambda sweep vs baselines — {title_suffix}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=7, ncol=2, loc="lower right")
    ax.grid(alpha=0.3)
    ax.set_xlim(0.05, 1.05)

plt.tight_layout()
save_path = ANALYSIS_DIR / "03_lambda_vs_map.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


**Figure 3 · Lambda-vs-mAP Curve (The Key Plot)**

*x-axis:* AEIoU lambda (0.1 to 1.0) · *y-axis:* mAP · *Horizontal lines:* baselines
*Solid/dashed/dotted:* clean / low / high splits

**This is the most important figure.** If the AEIoU curve exceeds ALL baseline
reference lines at some lambda, AEIoU is the best loss for polyp detection.
The peak lambda is the recommended setting. If it matches notebook 02 (DUO),
that is strong evidence of a domain-agnostic optimal lambda.


In [ ]:
# --- Fig 4: Lambda sensitivity heatmap (mAP50-95 and mAP50 side by side)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [
    (axes[0], "map95_mean", "mAP50-95"),
    (axes[1], "map50_mean", "mAP50"),
]:
    heat_data = []
    for r in AEIOU_RIGIDITIES:
        loss_name = f"aeiou_r{_fmt_r(r)}"
        row_data = {"lambda": r}
        for split in ["clean", "low", "high"]:
            sub = df_agg[(df_agg["loss"] == loss_name) & (df_agg["split"] == split)]
            row_data[split] = sub[metric].values[0] if not sub.empty else float("nan")
        heat_data.append(row_data)

    heat_df = pd.DataFrame(heat_data).set_index("lambda")
    sns.heatmap(heat_df.T, annot=True, fmt=".4f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, cbar_kws={"label": title})
    ax.set_title(f"AEIoU lambda sensitivity — {title}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Rigidity (lambda)", fontsize=10)
    ax.set_ylabel("Split", fontsize=10)

plt.tight_layout()
save_path = ANALYSIS_DIR / "04_lambda_heatmap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

# Report optimal lambda per split
for split in ["clean", "low", "high"]:
    sub = df_agg[df_agg["split"] == split]
    aeiou_sub = sub[sub["loss"].str.startswith("aeiou")]
    if not aeiou_sub.empty:
        best_row = aeiou_sub.loc[aeiou_sub["map95_mean"].idxmax()]
        print(f"  Best lambda for {split}: {best_row['loss']}  mAP95={best_row['map95_mean']:.4f}")
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 5: Learning curves — train loss + val mAP for ALL losses (clean split)
import matplotlib.pyplot as plt

MAP95_COL = "metrics/mAP50-95(B)"
LOSS_COL  = "train/box_loss"

# Show baselines + select AEIoU (0.1, 0.3, 0.5, 1.0) for readability
vis_losses = BASELINE_LOSS_NAMES + ["aeiou_r0p1", "aeiou_r0p3", "aeiou_r0p5", "aeiou_r1p0"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("Learning curves — clean split (all losses)", fontsize=13, fontweight="bold")

for loss_name in vis_losses:
    color = PALETTE.get(loss_name, "#888")
    label = LOSS_LABELS.get(loss_name, loss_name)

    # Average across seeds for this loss on clean split
    matching = df_all[(df_all["loss"] == loss_name) & (df_all["split"] == "clean")]
    if matching.empty:
        continue
    avg = matching.groupby("epoch").mean(numeric_only=True).reset_index()

    lw = 2.5 if loss_name in ["eiou", "eciou"] or "aeiou" in loss_name else 1.2
    ls = "-" if loss_name in BASELINE_LOSS_NAMES else "--"

    if LOSS_COL in avg.columns:
        axes[0].plot(avg["epoch"], avg[LOSS_COL], color=color, lw=lw, ls=ls, label=label)
    if MAP95_COL in avg.columns:
        axes[1].plot(avg["epoch"], avg[MAP95_COL], color=color, lw=lw, ls=ls, label=label)

axes[0].set_ylabel("Training box loss"); axes[0].legend(fontsize=7, ncol=3); axes[0].grid(alpha=0.3)
axes[1].set_ylabel("Val mAP50-95"); axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=7, ncol=3); axes[1].grid(alpha=0.3)

plt.tight_layout()
save_path = ANALYSIS_DIR / "05_learning_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 6: Convergence speed — epoch to reach 90% of final mAP
import matplotlib.pyplot as plt
import numpy as np

MAP95_COL = "metrics/mAP50-95(B)"
threshold = 0.90

all_losses = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
conv_data = {split: [] for split in ["clean", "low", "high"]}

for loss_name in all_losses:
    for split in ["clean", "low", "high"]:
        sub = df_all[(df_all["loss"] == loss_name) & (df_all["split"] == split)]
        if sub.empty or MAP95_COL not in sub.columns:
            conv_data[split].append(EPOCHS)
            continue
        avg = sub.groupby("epoch")[MAP95_COL].mean().reset_index().sort_values("epoch")
        final_map = avg[MAP95_COL].iloc[-1]
        target = threshold * final_map
        reached = avg[avg[MAP95_COL] >= target]["epoch"]
        conv_epoch = int(reached.iloc[0]) if len(reached) else EPOCHS
        conv_data[split].append(conv_epoch)

x_pos = np.arange(len(all_losses))
display_labels = [LOSS_LABELS.get(l, l) for l in all_losses]
width = 0.25
fig, ax = plt.subplots(figsize=(16, 5))
for i, (split, color) in enumerate(zip(["clean","low","high"],
                                        ["#4CAF50","#FFC107","#F44336"])):
    ax.bar(x_pos + i*width, conv_data[split], width=width, color=color, alpha=0.85, label=split)

ax.axvline(x=len(BASELINE_LOSS_NAMES) - 0.5, color="black", linestyle=":", lw=1.5, alpha=0.4)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(display_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel(f"Epoch to reach {threshold*100:.0f}% of final mAP", fontsize=11)
ax.set_title("Convergence speed: epochs to 90% final mAP", fontsize=13, fontweight="bold")
ax.legend(title="Split")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "06_convergence_speed.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 7: Noise robustness gap (mAP_clean - mAP_high) — smaller = more robust
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

all_losses = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
gaps = []
for loss_name in all_losses:
    clean_map = pivot95.loc[loss_name, "clean"] if loss_name in pivot95.index else 0
    high_map  = pivot95.loc[loss_name, "high"]  if loss_name in pivot95.index else 0
    gaps.append(clean_map - high_map)

display_labels = [LOSS_LABELS.get(l, l) for l in all_losses]

# Color: baselines in palette, AEIoU in blue gradient
bar_colors = [PALETTE.get(l, "#888") for l in all_losses]

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(display_labels, gaps, color=bar_colors, edgecolor="white", lw=0.5)

# Mark EIoU reference line
eiou_idx = all_losses.index("eiou") if "eiou" in all_losses else 0
ax.axhline(y=gaps[eiou_idx], color="#E63946", linestyle="--", lw=1.5,
           label=f"EIoU gap = {gaps[eiou_idx]:.4f}")

ax.set_ylabel("mAP gap (clean - high) — smaller = more robust", fontsize=11)
ax.set_title("Noise Robustness Gap — All Losses", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{gap:.3f}", ha="center", va="bottom", fontsize=6)
plt.tight_layout()
save_path = ANALYSIS_DIR / "07_noise_robustness_gap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

# Ranking
ranked = sorted(zip(display_labels, gaps), key=lambda x: x[1])
print("\nNoise robustness ranking (smaller = more robust):")
for rank, (name, gap) in enumerate(ranked, 1):
    print(f"  {rank:2d}. {name:<20} gap={gap:.4f}")
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 8: Precision-Recall curves (EIoU vs ECIoU vs best AEIoU, clean split)
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# Identify best AEIoU by clean mAP50-95
aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"

losses_for_pr = ["eiou", "eciou", best_aeiou]
pr_colors = {"eiou": "#E63946", "eciou": "#9B2226", best_aeiou: "#00B4D8"}

fig, ax = plt.subplots(figsize=(8, 6))

for loss_name in losses_for_pr:
    # Use first seed for PR curves
    seed = SEEDS[0]
    run_name = f"kvasir_yolo26n_{loss_name}_clean_s{seed}_e{EPOCHS}"
    weights = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name} weights not found")
        continue

    model = YOLO(str(weights))
    val_res = model.val(data=str(SPLIT_CONFIGS["clean"]), verbose=False)

    # Extract PR data from Ultralytics results
    if hasattr(val_res.box, "curves") and "Precision" in val_res.box.curves:
        prec = val_res.box.curves["Precision"]
        rec  = val_res.box.curves["Recall"]
    elif hasattr(val_res.box, "p") and hasattr(val_res.box, "r"):
        prec = val_res.box.p.flatten()
        rec  = val_res.box.r.flatten()
    else:
        # Fallback: plot a simplified PR from multiple confidence thresholds
        print(f"  PR curve data not directly available for {loss_name}, using AP summary")
        continue

    color = pr_colors.get(loss_name, "#888")
    label = LOSS_LABELS.get(loss_name, loss_name)
    ax.plot(rec, prec, color=color, lw=2, label=f"{label} (AP={float(val_res.box.map50):.3f})")

ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision-Recall Curves (clean split)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
save_path = ANALYSIS_DIR / "08_pr_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 9: AP vs IoU threshold (EIoU vs ECIoU vs best AEIoU, clean split)
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"
THRESHOLD_RANGE = np.arange(0.50, 1.00, 0.05)

results_by_thresh = {}
for loss_name in ["eiou", "eciou", best_aeiou]:
    seed = SEEDS[0]
    run_name = f"kvasir_yolo26n_{loss_name}_clean_s{seed}_e{EPOCHS}"
    weights = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name}")
        continue
    model = YOLO(str(weights))
    aps = []
    for thresh in THRESHOLD_RANGE:
        val_res = model.val(data=str(SPLIT_CONFIGS["clean"]), iou=float(thresh), verbose=False)
        ap_val = float(val_res.box.map50) if hasattr(val_res.box, "map50") else float(val_res.box.maps[0])
        aps.append(ap_val)
    results_by_thresh[loss_name] = aps

colors = {"eiou": "#E63946", "eciou": "#9B2226", best_aeiou: "#00B4D8"}
fig, ax = plt.subplots(figsize=(8, 5))
for loss_name in results_by_thresh:
    label = LOSS_LABELS.get(loss_name, loss_name)
    ax.plot(THRESHOLD_RANGE, results_by_thresh[loss_name],
            marker="o", color=colors.get(loss_name, "#888"), lw=2, label=label)

ax.set_xlabel("IoU threshold", fontsize=11)
ax.set_ylabel("AP at threshold", fontsize=11)
ax.set_title("AP vs IoU Threshold (clean split)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "09_ap_at_thresholds.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 10: Training box loss stability (rolling std, window=3)
import matplotlib.pyplot as plt
import pandas as pd

LOSS_COL = "train/box_loss"
WINDOW = 3

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"

# Compare EIoU, ECIoU, and best AEIoU on clean and high splits
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Training box loss stability (rolling std, window={WINDOW} epochs)",
             fontsize=13, fontweight="bold")

for ax, split in zip(axes, ["clean", "high"]):
    for loss_name in ["eiou", "eciou", best_aeiou]:
        color = PALETTE.get(loss_name, "#888")
        label = LOSS_LABELS.get(loss_name, loss_name)
        sub = df_all[(df_all["loss"] == loss_name) & (df_all["split"] == split)]
        if sub.empty or LOSS_COL not in sub.columns:
            continue
        avg = sub.groupby("epoch")[LOSS_COL].mean().reset_index().sort_values("epoch")
        loss_vals = avg[LOSS_COL].values
        epochs_arr = avg["epoch"].values
        rolling_std = pd.Series(loss_vals).rolling(WINDOW, min_periods=1).std().values
        ax.plot(epochs_arr, loss_vals, color=color, lw=2, label=label)
        ax.fill_between(epochs_arr,
                        loss_vals - rolling_std, loss_vals + rolling_std,
                        color=color, alpha=0.15)
    ax.set_title(f"Split: {split}", fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("box_loss")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
save_path = ANALYSIS_DIR / "10_training_stability.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 11: Box dimension calibration scatter (pred W,H vs GT W,H)
# For all 200 val images: compare predicted and GT box dimensions.
# Perfect calibration = points on y=x diagonal.
import matplotlib.pyplot as plt
import numpy as np
import cv2
from ultralytics import YOLO
from pathlib import Path

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"

val_img_dir = DATASET_ROOT / "valid" / "images"
val_lbl_dir = DATASET_ROOT / "valid" / "labels"

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Box Dimension Calibration: Predicted vs GT (clean split)",
             fontsize=13, fontweight="bold")

for ax, loss_name in zip(axes, ["eiou", "eciou", best_aeiou]):
    seed = SEEDS[0]
    run_name = f"kvasir_yolo26n_{loss_name}_clean_s{seed}_e{EPOCHS}"
    weights = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        ax.set_title(f"{LOSS_LABELS.get(loss_name, loss_name)} — weights missing")
        continue
    model = YOLO(str(weights))

    gt_wh, pred_wh = [], []
    for lbl_path in sorted(val_lbl_dir.glob("*.txt")):
        img_path = val_img_dir / f"{lbl_path.stem}.jpg"
        if not img_path.exists(): continue
        # GT box
        line = lbl_path.read_text().strip().split("\n")[0]
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        gt_w, gt_h = bw * W, bh * H
        gt_wh.append((gt_w, gt_h))
        # Pred box
        res = model(str(img_path), verbose=False)[0]
        if res.boxes and len(res.boxes.xyxy):
            pb = res.boxes.xyxy.cpu().numpy()[0]
            pw, ph = pb[2] - pb[0], pb[3] - pb[1]
            pred_wh.append((pw, ph))
        else:
            pred_wh.append((0, 0))

    gt_wh = np.array(gt_wh)
    pred_wh = np.array(pred_wh)
    # Scatter W and H together
    ax.scatter(gt_wh[:, 0], pred_wh[:, 0], alpha=0.5, s=15, c="#2A9D8F", label="Width")
    ax.scatter(gt_wh[:, 1], pred_wh[:, 1], alpha=0.5, s=15, c="#E63946", label="Height")
    max_val = max(gt_wh.max(), pred_wh.max()) * 1.1
    ax.plot([0, max_val], [0, max_val], "k--", lw=1, label="y=x (perfect)")
    ax.set_xlabel("GT dimension (px)"); ax.set_ylabel("Predicted dimension (px)")
    ax.set_title(LOSS_LABELS.get(loss_name, loss_name), fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_aspect("equal")

plt.tight_layout()
save_path = ANALYSIS_DIR / "11_box_calibration.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


**Figure 11 · Box Dimension Calibration**

*x-axis:* GT box dimension (W or H in pixels) · *y-axis:* Predicted dimension
*Dashed line:* y=x (perfect calibration)

**Expected pattern:** AEIoU points should cluster tighter to the diagonal,
especially for small polyps (bottom-left). EIoU may show systematic over-estimation
of box size on small objects because the enclosing-box normaliser under-penalises
size errors when the enclosing box is large.


In [ ]:
# --- Fig 12: Statistical significance tests
# Paired Wilcoxon signed-rank test on per-image prediction IoU
# comparing EIoU vs best AEIoU across all validation images.
import numpy as np
from ultralytics import YOLO
from scipy import stats
import cv2

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"

val_img_dir = DATASET_ROOT / "valid" / "images"
val_lbl_dir = DATASET_ROOT / "valid" / "labels"

def _compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter + 1e-7)

# Collect per-image IoUs for each loss
per_image_ious = {}
for loss_name in ["eiou", "eciou", best_aeiou]:
    seed = SEEDS[0]
    run_name = f"kvasir_yolo26n_{loss_name}_clean_s{seed}_e{EPOCHS}"
    weights = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name}")
        continue
    model = YOLO(str(weights))
    ious = []
    for lbl_path in sorted(val_lbl_dir.glob("*.txt")):
        img_path = val_img_dir / f"{lbl_path.stem}.jpg"
        if not img_path.exists(): continue
        line = lbl_path.read_text().strip().split("\n")[0]
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        gt = np.array([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H])
        res = model(str(img_path), verbose=False)[0]
        if res.boxes and len(res.boxes.xyxy):
            pb = res.boxes.xyxy.cpu().numpy()[0]
            ious.append(_compute_iou(pb, gt))
        else:
            ious.append(0.0)
    per_image_ious[loss_name] = np.array(ious)
    print(f"{LOSS_LABELS.get(loss_name, loss_name)}: mean IoU = {np.mean(ious):.4f}, n = {len(ious)}")

# Wilcoxon signed-rank test: EIoU vs best AEIoU
print("\n=== Statistical Significance Tests (Wilcoxon signed-rank) ===")
if "eiou" in per_image_ious and best_aeiou in per_image_ious:
    stat, p = stats.wilcoxon(per_image_ious["eiou"], per_image_ious[best_aeiou])
    delta = np.mean(per_image_ious[best_aeiou]) - np.mean(per_image_ious["eiou"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"EIoU vs {best_aeiou}: stat={stat:.1f}, p={p:.6f} ({sig})")
    print(f"  Mean IoU delta: {delta:+.4f} (positive = AEIoU better)")

if "eiou" in per_image_ious and "eciou" in per_image_ious:
    stat, p = stats.wilcoxon(per_image_ious["eiou"], per_image_ious["eciou"])
    delta = np.mean(per_image_ious["eciou"]) - np.mean(per_image_ious["eiou"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    print(f"EIoU vs ECIoU: stat={stat:.1f}, p={p:.6f} ({sig})")
    print(f"  Mean IoU delta: {delta:+.4f} (positive = ECIoU better)")


In [ ]:
# --- Fig 13: Cross-dataset lambda consistency (Kvasir vs DUO)
# Loads optimal lambda from notebook 02's results if available, otherwise
# provides a stub for manual comparison.
import json

print("=== Cross-Dataset Lambda Consistency ===\n")

# This notebook's result
aeiou_clean = df_agg[(df_agg["split"] == "clean") & (df_agg["loss"].str.startswith("aeiou"))]
if not aeiou_clean.empty:
    kvasir_best = aeiou_clean.loc[aeiou_clean["map95_mean"].idxmax()]
    r_kvasir = kvasir_best["loss"].split("_r")[1].replace("p", ".")
    print(f"Kvasir-SEG optimal lambda: {r_kvasir}  (mAP95={kvasir_best['map95_mean']:.4f})")
else:
    r_kvasir = "?"
    print("Kvasir-SEG: no AEIoU results found")

# Try to load DUO results from notebook 02
duo_csv = PROJECT_DIR / "experiments" / "all_results_combined.csv"
if duo_csv.exists():
    import pandas as pd
    duo_df = pd.read_csv(duo_csv)
    if "loss" in duo_df.columns and "split" in duo_df.columns:
        duo_df.columns = duo_df.columns.str.strip()
        duo_final = duo_df.groupby("run_name").last().reset_index()
        duo_aeiou = duo_final[(duo_final["loss"].str.startswith("aeiou")) & (duo_final["split"] == "clean")]
        if "metrics/mAP50-95(B)" in duo_aeiou.columns and not duo_aeiou.empty:
            duo_best_idx = duo_aeiou["metrics/mAP50-95(B)"].idxmax()
            duo_best = duo_aeiou.loc[duo_best_idx]
            r_duo = duo_best["loss"].split("_r")[1].replace("p", ".")
            print(f"DUO optimal lambda:       {r_duo}  (mAP95={duo_best['metrics/mAP50-95(B)']:.4f})")

            if r_kvasir == r_duo:
                print(f"\n*** MATCH: optimal lambda = {r_kvasir} on BOTH datasets ***")
                print("This supports a domain-agnostic default for amorphous objects.")
            else:
                print(f"\nDifferent optima: Kvasir={r_kvasir}, DUO={r_duo}")
                print("The optimal lambda may be dataset-dependent.")
        else:
            print("DUO: no AEIoU results in combined CSV")
    else:
        print("DUO: CSV format unexpected")
else:
    print(f"DUO results not found at {duo_csv}")
    print("Run notebook 02 first to enable cross-dataset comparison.")


In [ ]:
# --- Fig 14: Runtime overhead per loss function
import json
import pandas as pd

print("=== Runtime Overhead ===\n")

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    timing = []
    for run_name, meta in manifest.items():
        if meta.get("elapsed_sec") and meta.get("status") == "complete":
            timing.append({
                "loss": meta.get("loss", "?"),
                "split": meta.get("split", "?"),
                "elapsed_sec": meta["elapsed_sec"],
                "sec_per_epoch": meta["elapsed_sec"] / meta.get("epochs", EPOCHS),
            })

    if timing:
        timing_df = pd.DataFrame(timing)
        summary = timing_df.groupby("loss").agg(
            mean_sec=("elapsed_sec", "mean"),
            mean_sec_per_epoch=("sec_per_epoch", "mean"),
            n_runs=("elapsed_sec", "count"),
        ).sort_values("mean_sec")

        print(summary.round(1).to_string())
        print(f"\nFastest: {summary['mean_sec'].idxmin()} ({summary['mean_sec'].min():.1f}s)")
        print(f"Slowest: {summary['mean_sec'].idxmax()} ({summary['mean_sec'].max():.1f}s)")

        overhead = (summary["mean_sec"].max() - summary["mean_sec"].min()) / summary["mean_sec"].min() * 100
        print(f"Max overhead: {overhead:.1f}%")
        summary.to_csv(ANALYSIS_DIR / "runtime_overhead.csv")
        print(f"Saved -> {ANALYSIS_DIR / 'runtime_overhead.csv'}")
    else:
        print("No timing data in manifest.")
else:
    print("Manifest not found.")


In [ ]:
# --- Fig 15: Per-prediction IoU distribution (KDE, all val images)
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# Reuse per_image_ious from statistical test cell
fig, ax = plt.subplots(figsize=(8, 5))
x_range = np.linspace(0, 1, 200)

for loss_name in per_image_ious:
    data = per_image_ious[loss_name]
    if len(data) < 2:
        continue
    color = PALETTE.get(loss_name, "#888")
    label = LOSS_LABELS.get(loss_name, loss_name)
    kde = gaussian_kde(data, bw_method=0.15)
    ax.plot(x_range, kde(x_range), color=color, lw=2.5, label=label)
    ax.fill_between(x_range, kde(x_range), color=color, alpha=0.1)
    ax.axvline(np.mean(data), color=color, linestyle="--", lw=1,
               label=f"  mean={np.mean(data):.3f}")

ax.set_xlabel("IoU with ground truth", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Per-Prediction IoU Distribution (200 val images, clean split)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "15_iou_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Fig 16: Polyp-size stratified prediction quality (small/medium/large)
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

val_lbl_dir = DATASET_ROOT / "valid" / "labels"

# Stratify per-image IoUs by GT box area
areas = []
for lbl_path in sorted(val_lbl_dir.glob("*.txt")):
    line = lbl_path.read_text().strip().split("\n")[0]
    _, cx, cy, bw, bh = [float(v) for v in line.split()]
    areas.append((lbl_path.stem, bw * bh))
areas.sort(key=lambda x: x[1])
n = len(areas)
strata = {
    "small":  set(s for s, _ in areas[:n//3]),
    "medium": set(s for s, _ in areas[n//3:2*n//3]),
    "large":  set(s for s, _ in areas[2*n//3:]),
}

# Build stratified results using per_image_ious (indexed by lbl_path order)
lbl_stems = [lbl_path.stem for lbl_path in sorted(val_lbl_dir.glob("*.txt"))]
stratified = {}
for loss_name in per_image_ious:
    stratified[loss_name] = {}
    ious = per_image_ious[loss_name]
    for stratum, stems in strata.items():
        indices = [i for i, s in enumerate(lbl_stems) if s in stems and i < len(ious)]
        stratified[loss_name][stratum] = np.mean([ious[i] for i in indices]) if indices else 0

stratum_names = ["small", "medium", "large"]
x = np.arange(len(stratum_names))
n_losses = len(per_image_ious)
w = 0.8 / max(n_losses, 1)
fig, ax = plt.subplots(figsize=(10, 5))

for i, loss_name in enumerate(per_image_ious):
    color = PALETTE.get(loss_name, "#888")
    label = LOSS_LABELS.get(loss_name, loss_name)
    vals = [stratified[loss_name][s] for s in stratum_names]
    ax.bar(x + i*w - (n_losses-1)*w/2, vals, w, color=color, alpha=0.85, label=label)

ax.set_xticks(x)
ax.set_xticklabels(["Small polyps\n(bottom 33%)", "Medium polyps\n(middle 33%)", "Large polyps\n(top 33%)"])
ax.set_ylabel("Mean prediction IoU", fontsize=11)
ax.set_title("Polyp-Size Stratified Prediction Quality", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "16_size_stratified.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


**Figure 16 · Size-Stratified Prediction Quality**

**Expected pattern:** AEIoU should gain most on **small polyps** — these have
highly irregular boundaries relative to their size, making EIoU's enclosing-box
normaliser overly lenient. AEIoU's target-dim normaliser scales the penalty
to the polyp itself, so small polyps get proportionally correct gradients.


In [ ]:
# --- Fig 17: Failure case analysis — images where EIoU beats best AEIoU
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2, numpy as np

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"

if "eiou" not in per_image_ious or best_aeiou not in per_image_ious:
    print("Skipping failure cases — per-image IoU data missing.")
else:
    eiou_ious  = per_image_ious["eiou"]
    aeiou_ious = per_image_ious[best_aeiou]

    # Find images where EIoU has higher IoU than AEIoU
    val_lbl_dir = DATASET_ROOT / "valid" / "labels"
    lbl_stems = [p.stem for p in sorted(val_lbl_dir.glob("*.txt"))]

    delta = eiou_ious - aeiou_ious
    failure_indices = np.argsort(delta)[::-1][:5]  # top 5 where EIoU > AEIoU

    if np.all(delta[failure_indices] <= 0):
        print("No failure cases found — AEIoU >= EIoU on all images!")
    else:
        fig, axes = plt.subplots(1, min(5, len(failure_indices)), figsize=(20, 4))
        if not hasattr(axes, "__len__"): axes = [axes]

        fig.suptitle(f"Failure Cases: Images Where EIoU Beats {best_aeiou}",
                     fontsize=13, fontweight="bold")

        for ax, idx in zip(axes, failure_indices):
            if idx >= len(lbl_stems):
                continue
            stem = lbl_stems[idx]
            img_path = DATASET_ROOT / "valid" / "images" / f"{stem}.jpg"
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f"EIoU: {eiou_ious[idx]:.3f}  |  AEIoU: {aeiou_ious[idx]:.3f}\n"
                         f"delta = {delta[idx]:+.3f}", fontsize=8)
            ax.axis("off")

        plt.tight_layout()
        save_path = ANALYSIS_DIR / "17_failure_cases.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved -> {save_path}")
        print(f"\nFailure analysis: EIoU wins on {np.sum(delta > 0)}/{len(delta)} images "
              f"({np.sum(delta > 0)/len(delta)*100:.1f}%)")


**Figure 17 · Failure Cases**

Shows the images where EIoU produces higher per-prediction IoU than AEIoU.
Understanding failure modes is critical for an honest paper:
- Large, well-defined polyps with clear boundaries may not benefit from
  reduced shape penalty — EIoU's full penalty is appropriate
- Round polyps (AR near 1.0) have reliable extent labels, so lambda<1 is unnecessary
- If failure cases are all large/round, that confirms the hypothesis: AEIoU
  helps on amorphous objects, not on regular ones


## Section 12 · Bounding Box Visualisation

Side-by-side comparison on 5 stratified demo images:
EIoU vs ECIoU vs best AEIoU, with GT box and per-prediction IoU labels.


In [ ]:
# --- Select 5 stratified demo images + run inference from key models
import cv2, numpy as np, pickle
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

val_img_dir = DATASET_ROOT / "valid" / "images"
val_lbl_dir = DATASET_ROOT / "valid" / "labels"

# Stratified selection by GT box area
records = []
for img_path in sorted(val_img_dir.glob("*.jpg")):
    lbl_path = val_lbl_dir / f"{img_path.stem}.txt"
    if not lbl_path.exists(): continue
    line = lbl_path.read_text().strip().split("\n")[0]
    _, cx, cy, bw, bh = [float(v) for v in line.split()]
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    x1=(cx-bw/2)*W; y1=(cy-bh/2)*H; x2=(cx+bw/2)*W; y2=(cy+bh/2)*H
    area = (x2-x1)*(y2-y1)
    records.append((img_path, (x1,y1,x2,y2), area))
records.sort(key=lambda r: r[2])
indices = np.linspace(0, len(records)-1, 5, dtype=int)
DEMO = [records[i] for i in indices]

aeiou_entries = pivot95[pivot95.index.str.startswith("aeiou")]
best_aeiou = aeiou_entries["clean"].idxmax() if not aeiou_entries.empty else "aeiou_r0p3"
vis_losses = ["eiou", "eciou", best_aeiou]

# Run inference
inference = {}
for loss_name in vis_losses:
    seed = SEEDS[0]
    run_name = f"kvasir_yolo26n_{loss_name}_clean_s{seed}_e{EPOCHS}"
    weights = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name}")
        continue
    model = YOLO(str(weights))
    preds = {}
    for img_path, gt, area in DEMO:
        res = model(str(img_path), verbose=False)[0]
        boxes = res.boxes.xyxy.cpu().numpy() if res.boxes else np.zeros((0,4))
        preds[img_path.name] = boxes
    inference[loss_name] = preds

# Draw comparison grid
n_rows, n_cols = len(DEMO), len(vis_losses)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
fig.suptitle("Bbox Comparison: EIoU vs ECIoU vs Best AEIoU",
             fontsize=13, fontweight="bold", y=1.01)

for row, (img_path, gt_box, area) in enumerate(DEMO):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    gt_np = np.array(gt_box)

    for col, loss_name in enumerate(vis_losses):
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.imshow(img)
        # GT box — white dashed
        x1,y1,x2,y2 = gt_np
        ax.add_patch(mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=1.5, edgecolor="white", facecolor="none", linestyle="--"))
        # Predictions
        color = PALETTE.get(loss_name, "#888")
        boxes = inference.get(loss_name, {}).get(img_path.name, np.zeros((0,4)))
        for pb in boxes:
            iou = _compute_iou(pb, gt_np)
            rx1,ry1,rx2,ry2 = pb
            ax.add_patch(mpatches.Rectangle((rx1,ry1), rx2-rx1, ry2-ry1,
                         linewidth=2, edgecolor=color, facecolor="none"))
            ax.text(rx1, ry1-5, f"IoU={iou:.2f}", color=color, fontsize=8,
                    fontweight="bold", bbox=dict(facecolor="black", alpha=0.6, pad=1))
        if row == 0:
            ax.set_title(LOSS_LABELS.get(loss_name, loss_name), fontsize=10, fontweight="bold", color=color)
        if col == 0:
            ax.set_ylabel(f"area={area:.0f}px2", fontsize=8)
        ax.axis("off")

plt.tight_layout()
save_path = ANALYSIS_DIR / "18_bbox_comparison.png"
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")


In [ ]:
# --- Noise robustness ranking table (all losses)
import pandas as pd

all_losses = BASELINE_LOSS_NAMES + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
ranking = []
for loss_name in all_losses:
    if loss_name not in pivot95.index:
        continue
    clean_map = pivot95.loc[loss_name, "clean"]
    low_map   = pivot95.loc[loss_name, "low"]
    high_map  = pivot95.loc[loss_name, "high"]
    ratio     = high_map / clean_map if clean_map > 0 else 0
    ranking.append({
        "loss": loss_name, "label": LOSS_LABELS.get(loss_name, loss_name),
        "clean": clean_map, "low": low_map, "high": high_map,
        "robust_ratio": ratio,
    })

rank_df = pd.DataFrame(ranking).sort_values("robust_ratio", ascending=False).reset_index(drop=True)
rank_df.index += 1

print("Noise Robustness Ranking (mAP_high / mAP_clean — higher = more robust):")
print(rank_df.to_string())
rank_df.to_csv(ANALYSIS_DIR / "robustness_ranking.csv")
champion = rank_df.iloc[0]
print(f"\nMost robust: {champion['label']}  ratio={champion['robust_ratio']:.4f}")


## Section 13 · Summary & Conclusions

### Key Findings

| Analysis | What It Tests | Supports AEIoU? |
|---|---|---|
| mAP50-95 (Fig 2) | Localisation quality vs ALL baselines | If AEIoU > all 6 baselines |
| mAP50 (Fig 2b) | Detection quality | If AEIoU improves detection, not just boxes |
| Lambda curve (Fig 3) | Optimal lambda and whether it beats all baselines | Peak above all reference lines |
| Lambda heatmap (Fig 4) | Stability of optimal lambda across noise | Same peak across splits |
| Learning curves (Fig 5) | Convergence behaviour | AEIoU learns faster or to higher mAP |
| Convergence (Fig 6) | Speed to 90% mAP | Lower bar = faster |
| Robustness (Fig 7) | mAP degradation under noise | Smaller gap = more robust |
| PR curves (Fig 8) | Detection-recall trade-off | Higher curve = better |
| AP@thresholds (Fig 9) | Consistency across IoU thresholds | Above baselines at all thresholds |
| Stability (Fig 10) | Training smoothness | Narrower band = smoother |
| Box calibration (Fig 11) | Dimensional accuracy | Tighter to y=x diagonal |
| Significance (Fig 12) | Statistical validity | p < 0.05 |
| Cross-dataset (Fig 13) | Domain-agnostic lambda | Same optimal lambda as DUO |
| Runtime (Fig 14) | Computational cost | < 5% overhead |
| IoU distribution (Fig 15) | Overall box quality | Right-shifted KDE |
| Size-stratified (Fig 16) | Where gains concentrate | Largest on small polyps |
| Failure cases (Fig 17) | Honesty — where does AEIoU fail? | Large/round polyps |

### Recommended settings for Kvasir-SEG polyp detection
```
loss = AEIoULoss(rigidity=<optimal_lambda>, reduction="none")
```

### For publication
- Set `SEEDS = [42, 123, 456]` and re-run to get mean +/- std
- All analyses automatically handle multi-seed aggregation
- The statistical significance test uses paired Wilcoxon on per-image IoU


In [ ]:
# --- Save all artifacts and print manifest summary
import json, os

print("=== Experiment Artifact Summary ===\n")
print(f"Experiments dir: {EXPERIMENTS}")
print(f"Analysis dir:    {ANALYSIS_DIR}")

print("\nSaved figures:")
for f in sorted(ANALYSIS_DIR.glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>6.1f} KB")

print("\nSaved tables:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    complete = sum(1 for v in manifest.values() if v.get("status") == "complete")
    failed   = sum(1 for v in manifest.values() if v.get("status") == "failed")
    print(f"\nManifest: {len(manifest)} runs  |  {complete} complete  |  {failed} failed")

print("\nAll artifacts saved.")


In [ ]:
# --- Final sync: push analysis figures and manifest to Drive
import shutil

if DRIVE_AVAILABLE:
    drive_analysis = DRIVE_EXPERIMENTS / "analysis"
    drive_analysis.mkdir(parents=True, exist_ok=True)
    if ANALYSIS_DIR.exists():
        shutil.copytree(str(ANALYSIS_DIR), str(drive_analysis), dirs_exist_ok=True)
        n_figs = len(list(drive_analysis.glob("*.png")))
        print(f"Analysis figures synced to Drive: {n_figs} PNGs")

    for fname in ["manifest.json", "all_results_combined.csv"]:
        src = EXPERIMENTS / fname
        if src.exists():
            shutil.copy2(str(src), str(DRIVE_EXPERIMENTS / fname))
            print(f"  Synced {fname}")

    print(f"\nAll artifacts backed up to: {DRIVE_EXPERIMENTS}")
    n_total = sum(1 for _ in DRIVE_EXPERIMENTS.rglob("*") if _.is_file())
    print(f"Total files on Drive: {n_total}")
else:
    print("Drive not mounted — skipping final sync.")
    print("Tip: Run mount_drive() and re-run this cell to back up manually.")

print("\nNotebook 03 complete.")
